In [1]:
from pathlib import Path

from tapas_gmm.env.rlbench import RLBenchEnvironment, RLBenchEnvironmentConfig
from tapas_gmm.policy.gmm import GMMPolicy, GMMPolicyConfig
from tapas_gmm.policy.models.tpgmm import AutoTPGMMConfig

from rlbench.action_modes.arm_action_modes import BimanualEndEffectorPoseViaIK

import imageio.v2 as imageio


2026-07-16 14:40:13.390 | INFO     |  Running on cpu


/home/nils/Documents/Study Project/Code/riepybdlib/riepybdlib/data.py:34: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_listdir


In [2]:
model_path = Path(
    "../outputs/bimanual_tpgmm.pkl"
)

In [3]:
tapas_env = RLBenchEnvironment(
    RLBenchEnvironmentConfig(
        action_mode = BimanualEndEffectorPoseViaIK,
        robot_setup = "dual_panda",
        task = "BimanualDualPushButtons",
        cameras = ("front",),
        camera_pose = {},
        image_size = (128, 128),
        static = False,
        headless = False,
        scale_action = False,
        delay_gripper = False,
        gripper_plot = False,   
    )
)

policy = GMMPolicy(
    GMMPolicyConfig(
        suffix=None,
        model = AutoTPGMMConfig(),
        batch_predict_in_t_models = False,
        topp_in_t_models = False,
        binary_gripper_action = True,
        force_overwrite_checkpoint_config = True,
        pos_lag_thresh=0.03,
        quat_change_thresh=0.1,
    )
)

2026-07-16 14:40:30.808 | INFO     |  Initializing Policy:
2026-07-16 14:40:30.810 | INFO     |  No encoder config provided. Using None.
None


In [4]:
obs = tapas_env.reset()
policy.from_disk(str(model_path))
policy.eval()
policy.reset_episode(tapas_env)

2026-07-16 14:40:33.582 | INFO     |  Loading model:
2026-07-16 14:40:33.877 | ERROR    |  Config mismatch
root.tpgmm.reg_em_finish_diag 0.0002 != 0.001
root.tpgmm.reg_diag  0.0002 != 0.001
root.tpgmm.add_time_component True != False
root.tpgmm.add_action_component False != True
root.tpgmm.add_gripper_action True != False
root.tpgmm.reg_diag_gripper 0.02 != 0.1
root.tpgmm.reg_em_finish_diag_gripper 0.02 != 0.1
root.tpgmm.reg_init_diag 0.0005 != 5e-05
root.frame_selection.rel_score_threshold 0.1 != 0.75
root.demos_segmentation.components_prop_to_len True != False
root.demos_segmentation.distance_based False != True
root.demos_segmentation.velocity_based True != False

2026-07-16 14:40:33.878 | WARNING  |  Overwriting config. This can lead to unexpected errors.
2026-07-16 14:40:33.878 | INFO     |  Detected time-based model: True. Using time-driven policy. Set time_based in config to overwrite.
2026-07-16 14:40:33.878 | INFO     |  Creating local marginals
2026-07-16 14:40:33.878 | INFO 

In [5]:
total_reward = 0
frames = []

for step in range(450):
    action, info = policy.predict(obs)
    obs, reward, done, env_info = tapas_env.step(action)
    frame = obs.cameras["front"].rgb

    if frame.ndim == 4:
        frame = frame[0]

    if frame.shape[0] == 3:
        frame = frame.permute(1, 2, 0)

    frame = (frame * 255).clip(0, 255).byte().cpu().numpy()
    frames.append(frame)
    total_reward += reward

    if obs is None or done or info.get("done", False):
        break

imageio.mimsave("../outputs/bimanual_tpgmm_run.mp4", frames, fps=20)

print("steps:", step + 1)
print("total_reward:", total_reward)

tapas_env.close()

2026-07-16 14:40:33.899 | WARNING  |  Implementation lacking modulo rots, enforce z-up/down, etc.
2026-07-16 14:40:34.240 | INFO     |  Action [-2.03136256e-05 -8.75989006e-06  2.87464177e-04 -3.76984816e-05
  2.08225107e-04 -2.70923353e-04  9.99999941e-01  1.00000000e+00
  0.00000000e+00  8.49171341e-04 -1.69609065e-04 -1.42413018e-03
  1.55493877e-04 -3.94970771e-04 -6.73295749e-04  9.99999683e-01
  1.00000000e+00  0.00000000e+00]


/home/nils/Documents/Study Project/Code/TAPAS/tapas_gmm/utils/geometry_np.py:84: RuntimeWarning: invalid value encountered in arccos
  theta = 2 * np.arccos(2 * np.dot(q, r) ** 2 - 1)


2026-07-16 14:40:36.131 | INFO     |  Pos lag: [[ 6.10045657e-04  1.28607291e-04 -8.02854897e-04]
 [-1.39972612e-04 -5.33796152e-06 -2.43540344e-04]], quat lag: 0.0024373780263587264, pos change [[ 5.3626299e-04  1.4193356e-04 -3.7646294e-04]
 [ 2.2868812e-04 -3.4272671e-06  5.1772594e-04]], quat change 0.0013810679388648144
2026-07-16 14:40:36.132 | INFO     |  Action [ 7.71417073e-05 -5.12810522e-06 -2.72519351e-04 -1.16520822e-05
 -2.90901082e-04 -1.29727696e-04  9.99999949e-01  1.00000000e+00
  0.00000000e+00  4.06673243e-04 -7.62726890e-05 -9.33940990e-04
  5.63114457e-05 -5.35050142e-04 -2.63537313e-04  9.99999821e-01
  1.00000000e+00  0.00000000e+00]
2026-07-16 14:40:36.356 | INFO     |  Pos lag: [[ 3.33686375e-04  4.65654179e-05 -6.05368427e-04]
 [-2.43536719e-04  3.43437683e-06 -5.26081783e-04]], quat lag: 0.0021294998561750982, pos change [[ 2.75522470e-04  8.34390521e-05 -2.04205513e-04]
 [ 1.03488564e-04 -8.59797001e-06  2.79903412e-04]], quat change 0.0
2026-07-16 14:40:36

2026-07-16 14:40:37.097 | INFO     |  Pos lag: [[ 2.24715664e-04  5.62276471e-05 -7.25653344e-04]
 [-2.54829234e-04 -4.57616403e-06 -5.50902078e-04]], quat lag: 0.008632125222326611, pos change [[-2.6077032e-05  5.6289136e-05 -2.1648407e-04]
 [ 1.8745661e-05 -2.4810433e-05  1.3589859e-05]], quat change 0.012506107119957054
2026-07-16 14:40:37.100 | INFO     |  Action [ 1.35686455e-03 -5.99485478e-04  6.34668251e-03 -1.96961579e-02
 -3.63562347e-04 -1.61087430e-01  9.86743517e-01  1.00000000e+00
  0.00000000e+00 -6.56976874e-04 -3.79239452e-04 -1.67725822e-03
  8.99090679e-04  2.01281055e-02  7.39038234e-03  9.99769690e-01
  1.00000000e+00  0.00000000e+00]
2026-07-16 14:40:37.609 | INFO     |  Pos lag: [[-0.00022194  0.00033033 -0.0016734 ]
 [-0.00028992 -0.00026176 -0.00053082]], quat lag: 0.08461403137153947, pos change [[-4.8995018e-05  2.5361776e-05 -1.1229515e-04]
 [ 4.8904121e-04 -3.4178793e-04  7.0174932e-03]], quat change nan
2026-07-16 14:40:37.611 | INFO     |  Action [ 6.3794

2026-07-16 14:40:52.066 | INFO     |  Pos lag: [[ 3.50915268e-02  2.91978696e-03  5.75179319e-02]
 [ 4.13509524e-04  3.77843280e-05 -5.60446027e-04]], quat lag: 0.20773727015172086, pos change [[-0.01117668 -0.00634128  0.00048459]
 [-0.00460464 -0.00248152  0.02378035]], quat change 0.31171096197151793
2026-07-16 14:40:52.067 | INFO     |  Action [ 0.00161005 -0.12687991  0.04186795  0.53558475 -0.25739383 -0.80262144
  0.05192508  1.          0.          0.02156736 -0.00619334  0.03295819
  0.00216508  0.05622879 -0.02099991  0.99819469  0.          0.        ]


2026-07-16 14:40:52.703 | INFO     |  Pos lag: [[0.02520836 0.00650588 0.0323741 ]
 [0.11703179 0.05177738 0.03829287]], quat lag: 6.0770139064682125, pos change [[-4.7138035e-03  4.3563098e-03  5.6385994e-05]
 [-2.1591783e-05  3.4853816e-05  1.5366077e-04]], quat change 0.09935309079561773
2026-07-16 14:40:52.704 | INFO     |  Action [ 1.49883373e-03 -1.26867758e-01  4.17766486e-02  5.35114983e-01
 -2.57886880e-01 -8.02829261e-01  5.11027420e-02  1.00000000e+00
  0.00000000e+00  2.44772512e-02  3.69611468e-04  3.35728665e-02
 -2.44081525e-03  4.42401176e-02  2.42522825e-04  9.99017916e-01
  0.00000000e+00  0.00000000e+00]


2026-07-16 14:40:53.057 | INFO     |  Pos lag: [[0.02590952 0.00547326 0.0325453 ]
 [0.11700976 0.05178447 0.03832407]], quat lag: 6.078648363316259, pos change [[-6.9907308e-04  1.0317415e-03 -1.6433001e-04]
 [ 1.7583370e-05 -6.8098307e-06 -2.7418137e-05]], quat change 0.009056269996276352
2026-07-16 14:40:53.058 | INFO     |  Action [ 1.47980935e-03 -1.26851677e-01  4.18104029e-02  5.35125609e-01
 -2.57915616e-01 -8.02812416e-01  5.11110825e-02  1.00000000e+00
  0.00000000e+00  2.50384670e-02  1.56153146e-03  3.36317542e-02
 -2.78263380e-03  4.65005357e-02  7.42767714e-05  9.98914387e-01
  0.00000000e+00  0.00000000e+00]


2026-07-16 14:40:53.439 | INFO     |  Pos lag: [[0.02632253 0.00512071 0.03246533]
 [0.11701549 0.05176744 0.03831705]], quat lag: 6.078655670804178, pos change [[-4.1300058e-04  3.5254657e-04  7.9989433e-05]
 [-5.7369471e-06  1.7032027e-05  7.0333481e-06]], quat change 0.0036539624188174284
2026-07-16 14:40:53.441 | INFO     |  Action [-0.00581136 -0.12046187  0.0794903   0.38377977 -0.19591173 -0.87405153
 -0.22442285  1.          0.          0.03191936  0.0082009   0.06378198
 -0.00268021  0.04781853 -0.01289875  0.99876916  0.          0.        ]
2026-07-16 14:40:55.100 | INFO     |  Pos lag: [[0.03737519 0.00356022 0.06240851]
 [0.11052222 0.05261699 0.0766558 ]], quat lag: 5.377761310272575, pos change [[-2.0865202e-03 -2.1971762e-03  9.8228455e-05]
 [ 2.1308661e-06 -7.6889992e-06  2.6464462e-05]], quat change 0.05821871137559817
2026-07-16 14:40:55.102 | INFO     |  Action [-0.01693788 -0.11224001  0.08168653 -0.14456337  0.04278963 -0.64383956
 -0.75016072  0.          0.     

In [6]:
tapas_env.close()